In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

# Project folder
data_folder = Path(".")

# Saari CSV files find karo
csv_files = sorted(data_folder.glob("*.csv"))

print("Total CSV files found:", len(csv_files))

# Har CSV ko pandas DataFrame mein load karo
data = {}

for file in csv_files:
    name = file.stem
    data[name] = pd.read_csv(file)
    print(f"{name:30} {data[name].shape}")

Total CSV files found: 18
account_status_history         (60000, 8)
accounts                       (30000, 11)
agent_sessions                 (15000, 7)
agents                         (30000, 8)
borrowers                      (30600, 8)
call_attempts                  (120000, 9)
call_dispositions              (35000, 8)
calls                          (91350, 11)
campaigns                      (120, 7)
complaints                     (8000, 9)
daily_targeting                (45000, 7)
data_dictionary                (143, 3)
field_visits                   (25000, 10)
payments                       (25500, 9)
promises_to_pay                (18000, 9)
sms_events                     (45000, 8)
vendor_telephony               (15, 6)
whatsapp_events                (60600, 8)


In [4]:
# Data quality overview

overview = []

for name, df in data.items():
    overview.append({
        "table": name,
        "rows": len(df),
        "columns": len(df.columns),
        "missing_values": int(df.isna().sum().sum()),
        "duplicate_rows": int(df.duplicated().sum())
    })

overview_df = pd.DataFrame(overview)

overview_df

,table,rows,columns,missing_values,duplicate_rows
0,account_status_history,60000,8,0,0
1,accounts,30000,11,455,0
2,agent_sessions,15000,7,0,0
3,agents,30000,8,0,0
4,borrowers,30600,8,1509,600
5,call_attempts,120000,9,2400,0
6,call_dispositions,35000,8,0,0
7,calls,91350,11,1827,1271
8,campaigns,120,7,0,0
9,complaints,8000,9,0,0


In [5]:
# Har table ke column names dekho

for name, df in data.items():
    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)
    print(list(df.columns))


account_status_history
['history_id', 'account_id', 'borrower_id', 'event_at', 'status', 'changed_by', 'source', 'recorded_at']

accounts
['account_id', 'borrower_id', 'loan_type', 'principal_amount', 'outstanding_amount', 'dpd', 'risk_segment', 'status', 'opened_at', 'timezone', 'schema_version']

agent_sessions
['session_id', 'agent_id', 'login_at', 'channel', 'device_id', 'timezone', 'logout_at']

agents
['agent_id', 'employee_code', 'agent_name', 'vendor_id', 'team', 'status', 'joined_at', 'updated_at']

borrowers
['borrower_id', 'name', 'phone', 'email', 'city', 'created_at', 'updated_at', 'state']

call_attempts
['attempt_id', 'account_id', 'borrower_id', 'event_at', 'call_id', 'agent_id', 'attempt_no', 'vendor_id', 'attempt_status']

call_dispositions
['disposition_id', 'account_id', 'borrower_id', 'event_at', 'call_id', 'agent_id', 'disposition_code', 'disposition_version']

calls
['call_id', 'account_id', 'borrower_id', 'event_at', 'agent_id', 'campaign_id', 'direction', 'ven

In [6]:
# All tables: columns in a compact format

schema_df = pd.DataFrame([
    {
        "table": name,
        "columns": ", ".join(df.columns)
    }
    for name, df in data.items()
])

schema_df

,table,columns
0,account_status_history,"history_id, account_id, borrower_id, event_at,..."
1,accounts,"account_id, borrower_id, loan_type, principal_..."
2,agent_sessions,"session_id, agent_id, login_at, channel, devic..."
3,agents,"agent_id, employee_code, agent_name, vendor_id..."
4,borrowers,"borrower_id, name, phone, email, city, created..."
5,call_attempts,"attempt_id, account_id, borrower_id, event_at,..."
6,call_dispositions,"disposition_id, account_id, borrower_id, event..."
7,calls,"call_id, account_id, borrower_id, event_at, ag..."
8,campaigns,"campaign_id, campaign_name, channel, strategy_..."
9,complaints,"complaint_id, account_id, borrower_id, event_a..."


In [7]:
# Data dictionary dekho

data_dictionary = data["data_dictionary"]

data_dictionary.head(20)

,dataset,column,dtype
0,borrowers,borrower_id,object
1,borrowers,name,object
2,borrowers,phone,object
3,borrowers,email,object
4,borrowers,city,object
5,borrowers,created_at,datetime64[ns]
6,borrowers,updated_at,datetime64[ns]
7,borrowers,state,object
8,accounts,account_id,object
9,accounts,borrower_id,object


In [8]:
print("Total dictionary entries:", len(data_dictionary))
print("Datasets covered:", data_dictionary["dataset"].nunique())

data_dictionary.groupby("dataset").size().sort_values(ascending=False)

Total dictionary entries: 143
Datasets covered: 17


dataset
calls                     11
accounts                  11
field_visits              10
complaints                 9
promises_to_pay            9
payments                   9
call_attempts              9
account_status_history     8
sms_events                 8
whatsapp_events            8
call_dispositions          8
borrowers                  8
agents                     8
daily_targeting            7
agent_sessions             7
campaigns                  7
vendor_telephony           6
dtype: int64

In [9]:
# Unique ID / Primary Key investigation

id_report = []

for name, df in data.items():
    id_columns = [col for col in df.columns if col.endswith("_id")]
    
    for col in id_columns:
        id_report.append({
            "table": name,
            "id_column": col,
            "total_rows": len(df),
            "unique_values": df[col].nunique(),
            "duplicates": df[col].duplicated().sum(),
            "missing": df[col].isna().sum()
        })

id_report_df = pd.DataFrame(id_report)

id_report_df

,table,id_column,total_rows,unique_values,duplicates,missing
0,account_status_history,history_id,60000,60000,0,0
1,account_status_history,account_id,60000,25999,34001,0
2,account_status_history,borrower_id,60000,11916,48084,0
3,accounts,account_id,30000,30000,0,0
4,accounts,borrower_id,30000,10943,19056,455
5,agent_sessions,session_id,15000,15000,0,0
6,agent_sessions,agent_id,15000,1000,14000,0
7,agent_sessions,device_id,15000,1500,13500,0
8,agents,agent_id,30000,1000,29000,0
9,agents,vendor_id,30000,15,29985,0


In [10]:
# Exact duplicate records ka detailed check

duplicate_report = []

for name, df in data.items():
    exact_duplicates = df.duplicated().sum()
    
    duplicate_report.append({
        "table": name,
        "total_rows": len(df),
        "exact_duplicate_rows": int(exact_duplicates),
        "duplicate_percentage": round(
            exact_duplicates / len(df) * 100, 2
        )
    })

duplicate_report_df = pd.DataFrame(duplicate_report)

duplicate_report_df.sort_values(
    "exact_duplicate_rows",
    ascending=False
)

,table,total_rows,exact_duplicate_rows,duplicate_percentage
7,calls,91350,1271,1.39
17,whatsapp_events,60600,600,0.99
4,borrowers,30600,600,1.96
13,payments,25500,486,1.91
10,daily_targeting,45000,0,0.00
16,vendor_telephony,15,0,0.00
15,sms_events,45000,0,0.00
14,promises_to_pay,18000,0,0.00
12,field_visits,25000,0,0.00
11,data_dictionary,143,0,0.00


In [11]:
# Payments table ke columns
print("Payments columns:")
print(list(data["payments"].columns))

print("\nFirst 10 exact duplicate payment rows:")
data["payments"][data["payments"].duplicated(keep=False)].head(10)

Payments columns:
['payment_id', 'account_id', 'borrower_id', 'event_at', 'payment_reference', 'amount', 'payment_status', 'payment_method', 'provider_id']

First 10 exact duplicate payment rows:


,payment_id,account_id,borrower_id,event_at,payment_reference,amount,payment_status,payment_method,provider_id
55,PAYMENT0000056,ACC0019495,BRW0009716,2026-02-24 09:33:29,TXN0000008612,77982.73,PENDING,NETBANKING,VND0000007
75,PAYMENT0000076,ACC0027928,BRW0010869,2026-02-05 17:14:30,TXN0000006417,43712.91,PENDING,UPI,VND0000014
107,PAYMENT0000108,ACC0017047,BRW0009175,2026-06-09 08:17:45,TXN0000041404,102830.42,SUCCESS,NACH,VND0000003
148,PAYMENT0000149,ACC0018444,BRW0005741,2026-07-25 03:04:14,TXN0000066565,96766.12,SUCCESS,CASH,VND0000013
198,PAYMENT0000199,ACC0000056,BRW0001872,2026-05-12 22:49:22,TXN0000069967,81814.00,SUCCESS,NETBANKING,VND0000012
254,PAYMENT0000255,ACC0001307,BRW0001928,2026-05-07 18:37:15,TXN0000016416,5970.90,PENDING,UPI,VND0000005
262,PAYMENT0000263,ACC0004813,BRW0004424,2026-02-03 14:14:28,TXN0000019152,25143.15,PENDING,CARD,VND0000014
310,PAYMENT0000311,ACC0010109,BRW0005134,2026-04-09 20:18:27,TXN0000013702,75456.24,REVERSED,NETBANKING,VND0000012
551,PAYMENT0000552,ACC0021942,BRW0011343,2026-08-06 18:56:06,TXN0000000009,11792.14,SUCCESS,UPI,VND0000012
552,PAYMENT0000553,ACC0026650,BRW0004833,2026-05-01 05:10:02,TXN0000001233,45817.54,REVERSED,NACH,VND0000002


In [12]:
# Duplicate payment IDs ki investigation

payments = data["payments"]

duplicate_payment_ids = (
    payments[payments["payment_id"].duplicated(keep=False)]
    .sort_values("payment_id")
)

print("Rows involved in duplicate payment IDs:",
      len(duplicate_payment_ids))

print("Unique duplicate payment IDs:",
      duplicate_payment_ids["payment_id"].nunique())

print("\nSample duplicate payment records:")

duplicate_payment_ids.head(20)

Rows involved in duplicate payment IDs: 1000
Unique duplicate payment IDs: 500

Sample duplicate payment records:


,payment_id,account_id,borrower_id,event_at,payment_reference,amount,payment_status,payment_method,provider_id
55,PAYMENT0000056,ACC0019495,BRW0009716,2026-02-24 09:33:29,TXN0000008612,77982.73,PENDING,NETBANKING,VND0000007
25035,PAYMENT0000056,ACC0019495,BRW0009716,2026-02-24 09:33:29,TXN0000008612,77982.73,PENDING,NETBANKING,VND0000007
75,PAYMENT0000076,ACC0027928,BRW0010869,2026-02-05 17:14:30,TXN0000006417,43712.91,PENDING,UPI,VND0000014
25408,PAYMENT0000076,ACC0027928,BRW0010869,2026-02-05 17:14:30,TXN0000006417,43712.91,PENDING,UPI,VND0000014
107,PAYMENT0000108,ACC0017047,BRW0009175,2026-06-09 08:17:45,TXN0000041404,102830.42,SUCCESS,NACH,VND0000003
25481,PAYMENT0000108,ACC0017047,BRW0009175,2026-06-09 08:17:45,TXN0000041404,102830.42,SUCCESS,NACH,VND0000003
148,PAYMENT0000149,ACC0018444,BRW0005741,2026-07-25 03:04:14,TXN0000066565,96766.12,SUCCESS,CASH,VND0000013
25099,PAYMENT0000149,ACC0018444,BRW0005741,2026-07-25 03:04:14,TXN0000066565,96766.12,SUCCESS,CASH,VND0000013
198,PAYMENT0000199,ACC0000056,BRW0001872,2026-05-12 22:49:22,TXN0000069967,81814.00,SUCCESS,NETBANKING,VND0000012
25330,PAYMENT0000199,ACC0000056,BRW0001872,2026-05-12 22:49:22,TXN0000069967,81814.00,SUCCESS,NETBANKING,VND0000012


In [13]:
# Exact duplicate payments ki financial impact

duplicate_rows = payments[payments.duplicated(keep="first")]

duplicate_amount = duplicate_rows["amount"].sum()

total_payment_amount = payments["amount"].sum()

print("Total payment amount including duplicates: ₹", round(total_payment_amount, 2))
print("Amount contributed by exact duplicate rows: ₹", round(duplicate_amount, 2))
print("Inflation percentage: ", round((duplicate_amount / total_payment_amount) * 100, 2), "%")

Total payment amount including duplicates: ₹ 1917258617.15
Amount contributed by exact duplicate rows: ₹ 37299019.8
Inflation percentage:  1.95 %


In [14]:
# Duplicate payments ka status-wise impact

duplicate_status = (
    duplicate_rows
    .groupby("payment_status")
    .agg(
        duplicate_rows=("payment_id", "size"),
        duplicate_amount=("amount", "sum")
    )
    .sort_values("duplicate_amount", ascending=False)
)

duplicate_status

,duplicate_rows,duplicate_amount
payment_status,,
SUCCESS,335,25011462.19
FAILED,66,5060989.58
PENDING,56,4502633.24
REVERSED,29,2723934.79


In [15]:
# SUCCESS payments mein exact duplicates ka impact

success_payments = payments[
    payments["payment_status"] == "SUCCESS"
]

success_duplicate_rows = duplicate_rows[
    duplicate_rows["payment_status"] == "SUCCESS"
]

total_success_amount = success_payments["amount"].sum()
duplicate_success_amount = success_duplicate_rows["amount"].sum()

print("Total SUCCESS payment amount: ₹", round(total_success_amount, 2))
print("Duplicate SUCCESS amount: ₹", round(duplicate_success_amount, 2))
print(
    "SUCCESS amount potentially inflated by:",
    round((duplicate_success_amount / total_success_amount) * 100, 2),
    "%"
)

Total SUCCESS payment amount: ₹ 1341485926.33
Duplicate SUCCESS amount: ₹ 25011462.19
SUCCESS amount potentially inflated by: 1.86 %


In [16]:
# Duplicate SUCCESS payment IDs ko verify karo

success_duplicate_check = (
    success_duplicate_rows
    .groupby("payment_id")
    .size()
    .value_counts()
    .sort_index()
)

print("SUCCESS duplicate payment IDs ka frequency:")
print(success_duplicate_check)

print("\nTotal unique SUCCESS duplicate payment IDs:",
      success_duplicate_rows["payment_id"].nunique())

SUCCESS duplicate payment IDs ka frequency:
1    335
Name: count, dtype: int64

Total unique SUCCESS duplicate payment IDs: 335


In [17]:
# Clean payments table - exact duplicate rows remove

payments_clean = payments.drop_duplicates().copy()

print("Original payment rows:", len(payments))
print("Clean payment rows:", len(payments_clean))
print("Rows removed:", len(payments) - len(payments_clean))

print("\nOriginal payment amount: ₹", round(payments["amount"].sum(), 2))
print("Clean payment amount: ₹", round(payments_clean["amount"].sum(), 2))
print("Amount removed: ₹", round(
    payments["amount"].sum() - payments_clean["amount"].sum(), 2
))

Original payment rows: 25500
Clean payment rows: 25014
Rows removed: 486

Original payment amount: ₹ 1917258617.15
Clean payment amount: ₹ 1879959597.35
Amount removed: ₹ 37299019.8


In [18]:
# Missing values ka detailed report

missing_report = []

for name, df in data.items():
    missing = df.isna().sum()
    
    for column, count in missing.items():
        if count > 0:
            missing_report.append({
                "table": name,
                "column": column,
                "missing_count": int(count),
                "missing_percentage": round(
                    count / len(df) * 100, 2
                )
            })

missing_df = pd.DataFrame(missing_report)

missing_df.sort_values(
    "missing_count",
    ascending=False
)

,table,column,missing_count,missing_percentage
3,call_attempts,vendor_id,2400,2.00
4,calls,agent_id,1827,2.00
2,borrowers,email,895,2.92
1,borrowers,phone,614,2.01
0,accounts,borrower_id,455,1.52
6,payments,payment_reference,382,1.50
5,field_visits,scheduled_at,250,1.00


In [19]:
# Missing payment references ko inspect karo

missing_payment_reference = payments[
    payments["payment_reference"].isna()
]

print("Missing payment_reference rows:",
      len(missing_payment_reference))

print("\nPayment status of missing references:")
print(
    missing_payment_reference["payment_status"]
    .value_counts(dropna=False)
)

print("\nSample rows:")
missing_payment_reference.head(10)

Missing payment_reference rows: 382

Payment status of missing references:
payment_status
SUCCESS     265
FAILED       55
PENDING      39
REVERSED     23
Name: count, dtype: int64

Sample rows:


,payment_id,account_id,borrower_id,event_at,payment_reference,amount,payment_status,payment_method,provider_id
12,PAYMENT0000013,ACC0026504,BRW0009822,2026-05-06 22:30:42,NaN,41543.11,SUCCESS,CARD,VND0000013
153,PAYMENT0000154,ACC0024849,BRW0009430,2026-04-02 22:17:21,NaN,73382.67,FAILED,UPI,VND0000005
189,PAYMENT0000190,ACC0008922,BRW0010315,2026-08-04 06:05:43,NaN,39739.80,SUCCESS,NACH,VND0000011
210,PAYMENT0000211,ACC0011320,BRW0004951,2026-07-30 11:36:50,NaN,18417.58,SUCCESS,CASH,VND0000001
276,PAYMENT0000277,ACC0012078,BRW0009370,2026-05-26 16:27:17,NaN,103032.48,SUCCESS,NETBANKING,VND0000010
413,PAYMENT0000414,ACC0029253,BRW0008819,2026-01-15 16:40:04,NaN,131125.68,SUCCESS,CASH,VND0000014
446,PAYMENT0000447,ACC0015233,BRW0009862,2026-04-26 23:17:16,NaN,93762.91,PENDING,UPI,VND0000003
513,PAYMENT0000514,ACC0013461,BRW0011267,2026-08-06 09:06:35,NaN,29075.40,FAILED,UPI,VND0000002
539,PAYMENT0000540,ACC0028470,BRW0002140,2026-06-11 11:25:18,NaN,89925.12,SUCCESS,CARD,VND0000009
554,PAYMENT0000555,ACC0026518,BRW0000130,2026-04-29 18:44:23,NaN,12072.89,PENDING,NETBANKING,VND0000001


In [21]:
# Missing values ka correct check

print("CALL ATTEMPTS - missing vendor_id")
print(data["call_attempts"]["vendor_id"].isna().sum())

print("\nCALLS - missing agent_id")
print(data["calls"]["agent_id"].isna().sum())

print("\nBORROWERS - missing phone/email")
print("Missing phone:", data["borrowers"]["phone"].isna().sum())
print("Missing email:", data["borrowers"]["email"].isna().sum())

print("\nACCOUNTS - missing borrower_id")
print(data["accounts"]["borrower_id"].isna().sum())

print("\nFIELD VISITS - missing scheduled_at")
print(data["field_visits"]["scheduled_at"].isna().sum())

CALL ATTEMPTS - missing vendor_id
2400

CALLS - missing agent_id
1827

BORROWERS - missing phone/email
Missing phone: 614
Missing email: 895

ACCOUNTS - missing borrower_id
455

FIELD VISITS - missing scheduled_at
250


In [22]:
# Golden Dataset ke liye clean copies create karo

golden_data = {}

for name, df in data.items():
    golden_data[name] = df.drop_duplicates().copy()

print("Golden dataset copies created:")
for name, df in golden_data.items():
    print(f"{name:25} {df.shape}")

Golden dataset copies created:
account_status_history    (60000, 8)
accounts                  (30000, 11)
agent_sessions            (15000, 7)
agents                    (30000, 8)
borrowers                 (30000, 8)
call_attempts             (120000, 9)
call_dispositions         (35000, 8)
calls                     (90079, 11)
campaigns                 (120, 7)
complaints                (8000, 9)
daily_targeting           (45000, 7)
data_dictionary           (143, 3)
field_visits              (25000, 10)
payments                  (25014, 9)
promises_to_pay           (18000, 9)
sms_events                (45000, 8)
vendor_telephony          (15, 6)
whatsapp_events           (60000, 8)


In [23]:
# Golden dataset mein remaining missing values check

golden_missing = []

for name, df in golden_data.items():
    missing = df.isna().sum()

    for column, count in missing.items():
        if count > 0:
            golden_missing.append({
                "table": name,
                "column": column,
                "missing_count": int(count),
                "missing_percentage": round(
                    count / len(df) * 100, 2
                )
            })

golden_missing_df = pd.DataFrame(golden_missing)

golden_missing_df.sort_values(
    "missing_count",
    ascending=False
)

,table,column,missing_count,missing_percentage
3,call_attempts,vendor_id,2400,2.00
4,calls,agent_id,1827,2.03
2,borrowers,email,880,2.93
1,borrowers,phone,604,2.01
0,accounts,borrower_id,455,1.52
6,payments,payment_reference,382,1.53
5,field_visits,scheduled_at,250,1.00


In [24]:
# Missing values ka business context check

print("1. CALL ATTEMPTS - missing vendor_id")
print(
    golden_data["call_attempts"]
    .loc[
        golden_data["call_attempts"]["vendor_id"].isna(),
        ["attempt_id", "account_id", "borrower_id", "call_id"]
    ]
    .head(10)
)

print("\n2. CALLS - missing agent_id")
print(
    golden_data["calls"]
    .loc[
        golden_data["calls"]["agent_id"].isna(),
        ["call_id", "account_id", "borrower_id", "event_at"]
    ]
    .head(10)
)

print("\n3. ACCOUNTS - missing borrower_id")
print(
    golden_data["accounts"]
    .loc[
        golden_data["accounts"]["borrower_id"].isna()
    ]
    .head(10)
)

1. CALL ATTEMPTS - missing vendor_id
         attempt_id  account_id borrower_id      call_id
57   ATTEMPT0000058  ACC0015316  BRW0004075  CALL0050802
97   ATTEMPT0000098  ACC0006250  BRW0010637  CALL0020804
136  ATTEMPT0000137  ACC0018081  BRW0002946  CALL0041440
152  ATTEMPT0000153  ACC0017387  BRW0002474  CALL0076306
285  ATTEMPT0000286  ACC0000103  BRW0010767  CALL0010459
353  ATTEMPT0000354  ACC0023026  BRW0004862  CALL0071558
397  ATTEMPT0000398  ACC0020056  BRW0008863  CALL0085480
399  ATTEMPT0000400  ACC0015573  BRW0003670  CALL0039917
452  ATTEMPT0000453  ACC0027690  BRW0005983  CALL0087747
458  ATTEMPT0000459  ACC0000690  BRW0001646  CALL0057899

2. CALLS - missing agent_id
         call_id  account_id borrower_id             event_at
44   CALL0000045  ACC0000606  BRW0009525  2026-02-16 06:10:11
71   CALL0000072  ACC0026297  BRW0007223  2026-04-12 16:38:52
126  CALL0000127  ACC0007803  BRW0002353  2026-03-09 02:45:31
180  CALL0000181  ACC0027971  BRW0002797  2026-05-27 04:35:

In [26]:
print("CALLS columns:")
print(golden_data["calls"].columns.tolist())

print("\nCALLS - missing agent_id")
print(
    golden_data["calls"]
    .loc[
        golden_data["calls"]["agent_id"].isna()
    ]
    .head(10)
)

CALLS columns:
['call_id', 'account_id', 'borrower_id', 'event_at', 'agent_id', 'campaign_id', 'direction', 'vendor_id', 'call_status', 'duration_sec', 'timezone']

CALLS - missing agent_id
         call_id  account_id borrower_id             event_at agent_id  \
44   CALL0000045  ACC0000606  BRW0009525  2026-02-16 06:10:11      NaN   
71   CALL0000072  ACC0026297  BRW0007223  2026-04-12 16:38:52      NaN   
126  CALL0000127  ACC0007803  BRW0002353  2026-03-09 02:45:31      NaN   
180  CALL0000181  ACC0027971  BRW0002797  2026-05-27 04:35:43      NaN   
196  CALL0000197  ACC0001323  BRW0000116  2026-03-22 16:54:18      NaN   
317  CALL0000318  ACC0001543  BRW0001539  2026-01-10 20:20:26      NaN   
339  CALL0000340  ACC0004911  BRW0003498  2026-04-05 01:20:53      NaN   
365  CALL0000366  ACC0012718  BRW0008423  2026-04-26 21:18:49      NaN   
369  CALL0000370  ACC0019350  BRW0000751  2026-04-05 14:46:02      NaN   
376  CALL0000377  ACC0008264  BRW0009294  2026-07-25 02:27:08      NaN

In [27]:
# Calls - missing agent_id details

calls_missing_agent = golden_data["calls"][
    golden_data["calls"]["agent_id"].isna()
]

print("Total calls with missing agent_id:", len(calls_missing_agent))

print("\nCall status of missing agent_id:")
print(calls_missing_agent["call_status"].value_counts())

print("\nVendor-wise missing agent_id:")
print(calls_missing_agent["vendor_id"].value_counts())

print("\nSample rows:")
calls_missing_agent.head(10)

Total calls with missing agent_id: 1827

Call status of missing agent_id:
call_status
NO_ANSWER    385
VOICEMAIL    374
ANSWERED     358
BUSY         357
FAILED       353
Name: count, dtype: int64

Vendor-wise missing agent_id:
vendor_id
VND0000001    137
VND0000008    133
VND0000009    133
VND0000007    130
VND0000012    130
VND0000005    127
VND0000014    125
VND0000010    124
VND0000013    123
VND0000015    120
VND0000006    116
VND0000011    115
VND0000002    112
VND0000003    110
VND0000004     92
Name: count, dtype: int64

Sample rows:


,call_id,account_id,borrower_id,event_at,agent_id,campaign_id,direction,vendor_id,call_status,duration_sec,timezone
44,CALL0000045,ACC0000606,BRW0009525,2026-02-16 06:10:11,NaN,CMP0000095,OUTBOUND,VND0000002,FAILED,156,Asia/Kolkata
71,CALL0000072,ACC0026297,BRW0007223,2026-04-12 16:38:52,NaN,CMP0000072,OUTBOUND,VND0000007,VOICEMAIL,596,UTC
126,CALL0000127,ACC0007803,BRW0002353,2026-03-09 02:45:31,NaN,CMP0000017,OUTBOUND,VND0000004,NO_ANSWER,748,UTC
180,CALL0000181,ACC0027971,BRW0002797,2026-05-27 04:35:43,NaN,CMP0000003,OUTBOUND,VND0000002,VOICEMAIL,35,Asia/Kolkata
196,CALL0000197,ACC0001323,BRW0000116,2026-03-22 16:54:18,NaN,CMP0000039,OUTBOUND,VND0000015,NO_ANSWER,352,Asia/Kolkata
317,CALL0000318,ACC0001543,BRW0001539,2026-01-10 20:20:26,NaN,CMP0000118,OUTBOUND,VND0000012,ANSWERED,107,Asia/Dubai
339,CALL0000340,ACC0004911,BRW0003498,2026-04-05 01:20:53,NaN,CMP0000020,OUTBOUND,VND0000006,BUSY,180,Asia/Dubai
365,CALL0000366,ACC0012718,BRW0008423,2026-04-26 21:18:49,NaN,CMP0000051,OUTBOUND,VND0000005,BUSY,160,UTC
369,CALL0000370,ACC0019350,BRW0000751,2026-04-05 14:46:02,NaN,CMP0000048,OUTBOUND,VND0000006,ANSWERED,378,Asia/Kolkata
376,CALL0000377,ACC0008264,BRW0009294,2026-07-25 02:27:08,NaN,CMP0000006,OUTBOUND,VND0000006,BUSY,469,Asia/Dubai


In [28]:
# Golden Dataset - remaining important missing values

print("BORROWERS - missing phone/email")
print("Missing phone:", golden_data["borrowers"]["phone"].isna().sum())
print("Missing email:", golden_data["borrowers"]["email"].isna().sum())

print("\nACCOUNTS - missing borrower_id")
print("Missing borrower_id:", golden_data["accounts"]["borrower_id"].isna().sum())

print("\nFIELD VISITS - missing scheduled_at")
print("Missing scheduled_at:", golden_data["field_visits"]["scheduled_at"].isna().sum())

print("\nCALL ATTEMPTS - missing vendor_id")
print("Missing vendor_id:", golden_data["call_attempts"]["vendor_id"].isna().sum())

BORROWERS - missing phone/email
Missing phone: 604
Missing email: 880

ACCOUNTS - missing borrower_id
Missing borrower_id: 455

FIELD VISITS - missing scheduled_at
Missing scheduled_at: 250

CALL ATTEMPTS - missing vendor_id
Missing vendor_id: 2400


In [29]:
# Final Golden Dataset validation

print("=== GOLDEN DATASET FINAL VALIDATION ===\n")

for name, df in golden_data.items():
    print(f"{name:25} Rows: {len(df):6} | Columns: {len(df.columns):2} | "
          f"Duplicates: {df.duplicated().sum():4} | Missing: {df.isna().sum().sum():5}")

=== GOLDEN DATASET FINAL VALIDATION ===

account_status_history    Rows:  60000 | Columns:  8 | Duplicates:    0 | Missing:     0
accounts                  Rows:  30000 | Columns: 11 | Duplicates:    0 | Missing:   455
agent_sessions            Rows:  15000 | Columns:  7 | Duplicates:    0 | Missing:     0
agents                    Rows:  30000 | Columns:  8 | Duplicates:    0 | Missing:     0
borrowers                 Rows:  30000 | Columns:  8 | Duplicates:    0 | Missing:  1484
call_attempts             Rows: 120000 | Columns:  9 | Duplicates:    0 | Missing:  2400
call_dispositions         Rows:  35000 | Columns:  8 | Duplicates:    0 | Missing:     0
calls                     Rows:  90079 | Columns: 11 | Duplicates:    0 | Missing:  1827
campaigns                 Rows:    120 | Columns:  7 | Duplicates:    0 | Missing:     0
complaints                Rows:   8000 | Columns:  9 | Duplicates:    0 | Missing:     0
daily_targeting           Rows:  45000 | Columns:  7 | Duplicates:   

In [30]:
# Save Golden Dataset as CSV files

import os

golden_folder = "golden_dataset"
os.makedirs(golden_folder, exist_ok=True)

for name, df in golden_data.items():
    file_path = os.path.join(golden_folder, f"{name}.csv")
    df.to_csv(file_path, index=False)

print("Golden Dataset saved successfully!")
print(f"Location: {golden_folder}/")
print(f"Total files created: {len(golden_data)}")

Golden Dataset saved successfully!
Location: golden_dataset/
Total files created: 18
